In [ ]:
# --- snatac-cre-benchmark repo config ---
# Override with:  export PROJECT_ROOT=/path/to/project ; export DATA_ROOT=$PROJECT_ROOT/output_so
import os
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", ".")
DATA_ROOT    = os.environ.get("DATA_ROOT",    os.path.join(PROJECT_ROOT, "output_so"))


# Clustering

In [ ]:
# 개별 H5AD 파일들을 하나의 파일로 병합
import snapatac2 as snap

# 병합에 사용할 파일 경로 목록 (sample1.h5ad ~ sample9.h5ad)
file_paths = [
    f"{DATA_ROOT}/aggr/sample{i}.h5ad"
    for i in range(1, 10)
]

# 병합된 데이터 저장 경로
output_path = f"{DATA_ROOT}/aggr/colon.h5ads"

# 병합 및 저장 시도
print("Merging and saving data...")
try:
    data = snap.AnnDataSet(
        adatas=[(f"sample{i+1}", path) for i, path in enumerate(file_paths)],
        filename=output_path
    )
    print(f"Merged data saved to {output_path}")
except Exception as e:
    print(f"Error during data merging: {e}")


# ------------------------------------------------------------------------------------
# [3] 병합된 .h5ads 파일이 정상적으로 로드되는지 확인
#  - snap.read 로 읽고, 기본 메타 정보(셀 수, 피처 수)를 확인
# ------------------------------------------------------------------------------------
#병합된 파일 속 내용 확인
import snapatac2 as snap

# 병합된 데이터 파일 경로
merged_file_path = f"{DATA_ROOT}/aggr/colon.h5ads"

# 데이터 로드 및 확인
try:
    print(f"Loading merged dataset from {merged_file_path}...")
    merged_data = snap.read(merged_file_path)
    print("Merged dataset successfully loaded.")
    print(f"Number of cells: {merged_data.n_obs}")
    print(f"Number of features: {merged_data.n_vars}")
except Exception as e:
    print(f"Error loading merged dataset: {e}")

In [ ]:
data

In [ ]:
snap.pp.select_features(data, n_features=200000)
# Spectral embedding
snap.tl.spectral(data)

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"
os.environ["RAYON_NUM_THREADS"]    = "1"  # (snapATAC2의 Rust 병렬화)
os.environ["MALLOC_ARENA_MAX"]     = "2"  # glibc arena 폭주 방지

#batch effect correction
snap.pp.harmony(data, batch="sample", use_dims=30, max_iter_harmony=20)

In [ ]:
# Clustering
snap.pp.knn(data, n_neighbors=10, use_dims=30, use_rep="X_spectral_harmony") # Using embeding from harmony
snap.tl.leiden(data, random_state=0)  # Leiden clustering
print("Leiden clustering completed.")

In [ ]:
# UMAP calculation
snap.tl.umap(data, use_rep="X_spectral_harmony", use_dims=30)

In [ ]:
# UMAP visualization - Skip this cell, will visualize using gene_matrix later
snap.pl.umap(data, interactive=False, color="leiden")

In [ ]:
# UMAP visualization
snap.pl.umap(data, interactive=False, color="sample")

# Cell Annotation

In [ ]:
%%time
gene_matrix = snap.pp.make_gene_matrix(data, snap.genome.hg38)
gene_matrix

In [ ]:
# Copy over UMAP embedding and leiden clustering results
gene_matrix.obsm["X_umap"] = data.obsm["X_umap"]
gene_matrix.obs["leiden"] = data.obs["leiden"]

In [ ]:
import scanpy as sc

# UMAP 그리기 (화면 출력 X)
sc.pl.umap(
    gene_matrix,
    use_raw=False,
    color="leiden",
    legend_loc="on data",
    show=False
)

In [ ]:
import scanpy as sc

# UMAP 그리기 (화면 출력 X)
sc.pl.umap(
    gene_matrix,
    use_raw=False,
    color="leiden",
    show=False
)

# PDF로 저장
import matplotlib.pyplot as plt
plt.savefig("umap_leiden.pdf", bbox_inches="tight")
plt.close()

In [ ]:
import scanpy as sc

# UMAP 그리기 (화면 출력 X)
sc.pl.umap(
    gene_matrix,
    use_raw=False,
    color="leiden",
    show=False
)

# PDF로 저장
import matplotlib.pyplot as plt
plt.savefig("umap_leiden.pdf", bbox_inches="tight")
plt.close()

In [ ]:
sc.pl.umap(gene_matrix, use_raw=False, color="sample")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
# 마커 유전자 정의
marker_genes = {
    "Atrial Cardiomyocytes": ["NPPA", "MYH6", "MYL7"],
    "Ventricular Cardiomyocytes": ["MYH7", "HEY2", "MYL2"],
    "Fibroblasts": ["DCN"],
    "Endothelial": ["EGFL7", "VWF"],
    "Smooth Muscle": ["GJA4", "TAGLN"],
    "Macrophages": ["CD163", "MS4A6A"],
    "Lymphocytes": ["IL7R", "THEMIS"],
    "Adipocytes": ["ADIPOQ", "CIDEA"],
    "Nervous Cells": ["NRXN3", "GPM6B"],
    "Endocardial-like Cells": ["NRG3", "NPR3"],
    "Myofibroblasts": ["MYH10"],
    "Arterial Smooth Muscle Cells": ["ACTA2", "TAGLN"],
    "Epiblast":["NANOG", "POU5F1", "SOX2", "KLF17", "TDGF1"],
    "Primitive Endoderm":["GATA6", "GATA4", "SOX17", "PDGFRA"],
    "Trophectoderm": ["GATA3", "GATA2", "KRT18", "TEAD3"],
    "Early Inner Cell Mass": ["RSPO3", "ARGFX", "PRDM14", "SOX2"]
}

# Dot Plot 그리기
sc.pl.dotplot(
    gene_matrix,  # 데이터 객체
    var_names=marker_genes,  # 마커 유전자
    groupby="leiden",  # 클러스터별 그룹
    standard_scale="var",  # 변수를 표준화
    dot_min=0.1,  # 최소 점 크기
    dot_max=1,  # 최대 점 크기
    cmap="Blues",  # 컬러맵 설정
    figsize=(12, 10)  # 그림 크기 조정
)

# 그림 표시
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt

def save_dotplot(dotplot_obj, filename="marker_dotplot.pdf", dpi=300):
    """
    Scanpy dotplot을 PDF로 안전하게 저장하는 함수 (모든 버전 호환)
    """
    try:
        # 최신 Scanpy (1.9 이상)
        fig = dotplot_obj.figure
    except AttributeError:
        # 구버전 (1.8 이하)
        fig = dotplot_obj["dotplot_ax"].figure

    save_path = os.path.abspath(filename)
    fig.savefig(save_path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    print(f"✅ PDF 저장 완료: {save_path}")


# 사용 예시 -------------------------------------------------
import scanpy as sc

marker_genes = {
    "Atrial Cardiomyocytes": ["NPPA", "MYH6", "MYL7"],
    "Ventricular Cardiomyocytes": ["MYH7", "HEY2", "MYL2"],
    "Fibroblasts": ["DCN"],
    "Endothelial": ["EGFL7", "VWF"],
    "Smooth Muscle": ["GJA4", "TAGLN"],
    "Macrophages": ["CD163", "MS4A6A"],
    "Lymphocytes": ["IL7R", "THEMIS"],
    "Adipocytes": ["ADIPOQ", "CIDEA"],
    "Nervous Cells": ["NRXN3", "GPM6B"],
    "Endocardial-like Cells": ["NRG3", "NPR3"],
    "Myofibroblasts": ["MYH10"],
    "Arterial Smooth Muscle Cells": ["ACTA2", "TAGLN"],
    "Epiblast": ["NANOG", "POU5F1", "SOX2", "KLF17", "TDGF1"],
    "Primitive Endoderm": ["GATA6", "GATA4", "SOX17", "PDGFRA"],
    "Trophectoderm": ["GATA3", "GATA2", "KRT18", "TEAD3"],
    "Early Inner Cell Mass": ["RSPO3", "ARGFX", "PRDM14", "SOX2"]
}

dotplot = sc.pl.dotplot(
    gene_matrix,
    var_names=marker_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.1,
    dot_max=1,
    cmap="Blues",
    figsize=(12, 10),
    show=False
)

save_dotplot(dotplot, "marker_dotplot.pdf")

In [ ]:
data.close()

# Update H5AD files(Add cluster information)

In [ ]:
import os
import snapatac2 as snap
import scanpy as sc
import pandas as pd

# 원본 per-sample h5ad들 (수정 대상)
file_paths = [
    f"{DATA_ROOT}/aggr/sample{i}.h5ad"
    for i in range(1, 10)
]

# 업데이트 저장 경로
aggr_update_dir = f"{DATA_ROOT}/aggr_update"
os.makedirs(aggr_update_dir, exist_ok=True)

# gene_matrix: 바코드 중복 제거
gene_matrix = gene_matrix[~gene_matrix.obs.index.duplicated(keep='first')].copy()

# gene_matrix 메타데이터 추출
leiden = gene_matrix.obs["leiden"].astype(str)
sample = gene_matrix.obs["sample"].astype(str)
cell_barcodes = gene_matrix.obs.index.astype(str)

# 클러스터 → 세포 유형 매핑
cluster_ctype_dic = {
    '2': "Atrial_Cardiomyocytes", '16': "Atrial_Cardiomyocytes", '20': "Atrial_Cardiomyocytes",
    '0': "Ventricular_Cardiomyocytes", '4': "Ventricular_Cardiomyocytes",
    '3': "Ventricular_Cardiomyocytes", '5': "Ventricular_Cardiomyocytes",
    '6': "Ventricular_Cardiomyocytes", '7': "Ventricular_Cardiomyocytes",
    '8': "Ventricular_Cardiomyocytes", '11': "Ventricular_Cardiomyocytes", '15': "Ventricular_Cardiomyocytes",
    '10': "Primitive_Endoderm", '13': "Primitive_Endoderm",
    '12': "Endothelial", '14': "Endothelial",
    '19': "Myofibroblasts", '21': "Nervous_Cells", '22': "Trophectoderm", '18': "Macrophages", '17': "Smooth_Muscle",
    '1': "Fibroblasts", '9': "Fibroblasts"
}
cell_types = leiden.map(lambda l: cluster_ctype_dic.get(l, "Unknown"))

# gene_matrix 기반 메타 테이블
df = pd.DataFrame(
    {"leiden": leiden, "sample": sample, "celltype": cell_types},
    index=cell_barcodes,
)

# per-sample h5ad 업데이트
updated_file_paths = []  # [(group_name, path), ...]
for h5ad_path in file_paths:
    try:
        print(f"Processing {h5ad_path} ...")
        ad = sc.read_h5ad(h5ad_path)

        # 바코드 중복 제거
        ad = ad[~ad.obs.index.duplicated(keep='first')].copy()

        # gene_matrix 메타를 현재 파일 바코드 순서로 정렬
        merged = df.reindex(ad.obs.index.astype(str))

        # 파일명에서 sample 이름 추출 (예: sample3)
        sample_name = os.path.splitext(os.path.basename(h5ad_path))[0]

        # 누락값 보정
        merged["celltype"] = merged["celltype"].fillna("Unknown").astype(str)
        merged["leiden"]   = merged["leiden"].fillna("NA").astype(str)
        merged["sample"]   = merged["sample"].fillna(sample_name).astype(str)

        # obs에 기록
        ad.obs["celltype"] = pd.Categorical(merged["celltype"].values)
        ad.obs["leiden"]   = pd.Categorical(merged["leiden"].values)
        ad.obs["sample"]   = pd.Categorical(merged["sample"].values)
        ad.obs["sample_cluster"] = ad.obs["celltype"].astype(str) + "." + ad.obs["sample"].astype(str)

        # 저장 (aggr_update/sampleX_updated.h5ad)
        updated_path = os.path.join(aggr_update_dir, f"{sample_name}_updated.h5ad")
        ad.write(updated_path)
        updated_file_paths.append((sample_name, updated_path))
        print(f"✅ Updated: {updated_path} (matched {merged['leiden'].notna().sum()} cells)")

    except Exception as e:
        print(f"❌ Error processing {h5ad_path}: {e}")

# 새로운 AnnDataSet(.h5ads) 생성
updated_dataset_path = os.path.join(aggr_update_dir, "colon_updated.h5ads")
snap.AnnDataSet(adatas=updated_file_paths, filename=updated_dataset_path)
print(f"✅ New AnnDataSet with celltype added: {updated_dataset_path}")

In [ ]:
data.close()

# Convert to Fragment file

In [ ]:
data = snap.AnnDataSet(adatas=updated_file_paths, filename=updated_dataset_path)
print(f"✅ New AnnDataSet with celltype added: {updated_dataset_path}")

In [ ]:
import pandas as pd
import scanpy as sc
import snapatac2 as snap
import os

# per-sample에서 sample_cluster 시리즈를 모아 합치기
labels = []
for name, path in updated_file_paths:
    ad = sc.read_h5ad(path)
    if "sample_cluster" not in ad.obs:
        raise ValueError(f"{path}에 sample_cluster가 없습니다.")
    labels.append(ad.obs["sample_cluster"].astype(str))

groupby_series = pd.concat(labels)   # index = 전체 셀 바코드

# 내보내기
dir_fragments = f"{DATA_ROOT}/fragments"
os.makedirs(dir_fragments, exist_ok=True)

snap.ex.export_fragments(
    data,
    groupby=groupby_series,   # ← 컬럼명 대신 시리즈를 직접 전달
    out_dir=dir_fragments,
    prefix="mcluster",
    suffix="_fragments.bed"
)

# Convert to Tn5 ins file (X)

In [ ]:
import os
import pandas as pd

input_dir  = f"{DATA_ROOT}/fragments" #fragments_2 -> fragments 
output_dir = f"{DATA_ROOT}/tn5_ins"
os.makedirs(output_dir, exist_ok=True)

for fname in os.listdir(input_dir):
    in_path = os.path.join(input_dir, fname)
    if not os.path.isfile(in_path):
        continue

    try:
        # 1) 읽기 (5컬럼 가정)
        df = pd.read_csv(in_path, sep="\t", header=None)
        if df.shape[1] != 5:
            print(f"[SKIP] {fname}: cols={df.shape[1]} (expected 5)")
            continue
        df.columns = ["chrom", "start", "end", "barcode", "count"]

        # 2) Tn5 삽입점(1bp): 좌(start+4), 우(end-5)
        left = df[["chrom", "start", "barcode", "count"]].copy()
        left["tn5_start"] = (left["start"] + 4).clip(lower=0)
        left["tn5_end"]   = left["tn5_start"] + 1
        left = left[["chrom", "tn5_start", "tn5_end", "barcode", "count"]]

        right = df[["chrom", "end", "barcode", "count"]].copy()
        right["tn5_start"] = (right["end"] - 5).clip(lower=0)
        right["tn5_end"]   = right["tn5_start"] + 1
        right = right[["chrom", "tn5_start", "tn5_end", "barcode", "count"]]

        tn5 = pd.concat([left, right], ignore_index=True)

        # 3) 출력 파일명: 'fragments' → 'tn5_ins'로만 치환
        out_name = fname.replace("fragments", "tn5_ins")
        out_path = os.path.join(output_dir, out_name)

        tn5.to_csv(out_path, sep="\t", header=False, index=False)
        print(f"[OK] {out_path} ({len(tn5):,} rows)")
    except Exception as e:
        print(f"[ERROR] {fname}: {e}")

# Cluster analysis

In [ ]:
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

# 실제 파일 있는 폴더로 설정 
base = f"{DATA_ROOT}/aggr_update"

# *_updated.h5ad 자동 수집
paths = sorted(glob.glob(os.path.join(base, "sample*_updated.h5ad")))
if not paths:
    raise FileNotFoundError(f"*_updated.h5ad 파일을 찾지 못했습니다: {base}")

# 읽고 합치기
ad_list, keys = [], []
for p in paths:
    a = sc.read_h5ad(p)
    # 필요한 obs 컬럼 확인/캐스팅
    for col in ["leiden", "celltype", "sample"]:
        if col not in a.obs:
            raise KeyError(f"{os.path.basename(p)}에 '{col}' 컬럼이 없습니다.")
        a.obs[col] = a.obs[col].astype(str)
    ad_list.append(a)
    keys.append(os.path.splitext(os.path.basename(p))[0].replace("_updated",""))

ad_all = ad.concat(ad_list, join="outer", label="sample", keys=keys, index_unique=None)

# 집계
obs = ad_all.obs
print("📊 클러스터별 cell 개수")
print(obs["leiden"].value_counts().sort_index(), "\n")

print("📊 세포타입별 cell 개수")
print(obs["celltype"].value_counts(), "\n")

print("📊 클러스터 × 세포타입 교차표")
print(pd.crosstab(obs["leiden"], obs["celltype"]), "\n")

print("📊 샘플 × 세포타입 교차표")
print(pd.crosstab(obs["sample"], obs["celltype"]))

# 다음에 편하게 쓰려고, 합쳐진 하나짜리 h5ad로 저장(선택)
out_merged = os.path.join(base, "colon_merged_from_updated.h5ad")
ad_all.write(out_merged)
print(f"\n✅ merged h5ad 저장: {out_merged}")

In [ ]:
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

# 실제 파일 있는 폴더로 설정 
base = f"{DATA_ROOT}/aggr_update"

# *_updated.h5ad 자동 수집
paths = sorted(glob.glob(os.path.join(base, "sample*_updated.h5ad")))
if not paths:
    raise FileNotFoundError(f"*_updated.h5ad 파일을 찾지 못했습니다: {base}")

# 읽고 합치기
ad_list, keys = [], []
for p in paths:
    a = sc.read_h5ad(p)
    # 필요한 obs 컬럼 확인/캐스팅
    for col in ["leiden", "celltype", "sample"]:
        if col not in a.obs:
            raise KeyError(f"{os.path.basename(p)}에 '{col}' 컬럼이 없습니다.")
        a.obs[col] = a.obs[col].astype(str)
    ad_list.append(a)
    keys.append(os.path.splitext(os.path.basename(p))[0].replace("_updated",""))

ad_all = ad.concat(ad_list, join="outer", label="sample", keys=keys, index_unique=None)

# 집계
obs = ad_all.obs
print("📊 클러스터별 cell 개수")
cluster_counts = obs["leiden"].value_counts().sort_index()
print(cluster_counts, "\n")

print("📊 세포타입별 cell 개수")
celltype_counts = obs["celltype"].value_counts()
print(celltype_counts, "\n")

print("📊 클러스터 × 세포타입 교차표")
cluster_celltype_table = pd.crosstab(obs["leiden"], obs["celltype"])
print(cluster_celltype_table, "\n")

print("📊 샘플 × 세포타입 교차표")
sample_celltype_table = pd.crosstab(obs["sample"], obs["celltype"])
print(sample_celltype_table)

# 📂 CSV로 저장
outdir = f"{DATA_ROOT}/aggr_update"
os.makedirs(outdir, exist_ok=True)

cluster_counts.to_csv(os.path.join(outdir, "cluster_counts.csv"))
celltype_counts.to_csv(os.path.join(outdir, "celltype_counts.csv"))
cluster_celltype_table.to_csv(os.path.join(outdir, "cluster_x_celltype.csv"))
sample_celltype_table.to_csv(os.path.join(outdir, "sample_x_celltype.csv"))

print(f"\n✅ CSV 파일로 저장 완료 (경로: {outdir})")